# Evaluación experimental — estrategias de almacenamiento de k-mers

Bases de Datos II · Proyecto del segundo corte

Compara dos estrategias de almacenamiento para los perfiles de k-mers, sobre una
muestra aleatoria de 30,000 secuencias ya cargadas (con k-mers ya calculados, para
aislar el costo de la *estrategia de almacenamiento* del costo de *calcular* los
k-mers):

- **A) Embebido** (estrategia de producción): un documento por secuencia en
  `sequences`, con `kmers.k1`…`kmers.k6` anidados.
- **B) Colección aparte**: los metadatos de la secuencia en `bench_meta` y los
  k-mers en una colección independiente `bench_kmers_alt`, referenciados por
  `sequence_id`.

Para cada estrategia se mide: tiempo de carga, espacio en disco, y el tiempo de
3 consultas representativas con y sin índice (`explain()`/`executionStats`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
import json
import experiment_storage as exp

docs = exp.get_sample()
sample_id = docs[0]["original_id"]
print(f"Muestra: {len(docs)} secuencias. Ejemplo: {sample_id}")

Muestra: 30000 secuencias. Ejemplo: PDB00109510


## Carga: embebido vs. colección aparte

In [2]:
t_embedded = exp.load_embedded(docs)
t_separate = exp.load_separate(docs)
print(f"Tiempo de carga EMBEBIDO:      {t_embedded:.2f} s")
print(f"Tiempo de carga APARTE:        {t_separate:.2f} s")

Tiempo de carga EMBEBIDO:      17.17 s
Tiempo de carga APARTE:        25.15 s


**Interpretación:** la estrategia *aparte* reparte la carga en dos
`bulk_write` (metadatos + k-mers) a colecciones más pequeñas cada una, mientras
que *embebido* escribe un solo documento grande por secuencia. En la práctica el
tiempo de ambas es del mismo orden de magnitud para esta muestra; la diferencia
real aparece en las consultas (ver abajo).

## Espacio en disco (`collStats`)

In [3]:
stats = exp.storage_stats()
print(json.dumps(stats, indent=2))

{
  "embebido_MB": 507.63,
  "embebido_indices_MB": 0.75,
  "aparte_meta_MB": 51.04,
  "aparte_kmers_MB": 313.26,
  "aparte_indices_MB": 0.93,
  "aparte_total_MB": 364.31
}


**Interpretación:** el total en disco es similar entre ambas estrategias
(~500 MB para 30,000 secuencias) porque en ambos casos la secuencia de
nucleótidos completa —el campo más pesado— se guarda una sola vez por documento
de metadatos; la diferencia entre estrategias está en el *overhead* de tener una
colección adicional (un `_id`/`sequence_id` e índice extra por documento), que
aquí representa una fracción pequeña del total.

## Consultas representativas (con y sin índice)

### Consulta 1 — Perfil de k-mers (k=6) de UNA secuencia dada (punto)

In [4]:
qr = exp.run_queries(sample_id)
print(json.dumps({k: v for k, v in qr.items() if k.startswith("Q1")}, indent=2))

{
  "Q1_perfil_kmers_embebido": {
    "con_indice": {
      "tiempo_s": 0.00111,
      "docsExamined": 1
    },
    "sin_indice": {
      "tiempo_s": 0.12902,
      "docsExamined": 30000
    }
  },
  "Q1_perfil_kmers_aparte": {
    "con_indice": {
      "tiempo_s": 0.00081,
      "docsExamined": 1
    },
    "sin_indice": {
      "tiempo_s": 0.03632,
      "docsExamined": 30000
    }
  }
}


**Interpretación:** con índice (`original_id` / `sequence_id`), ambas
estrategias resuelven la consulta en ~1-2 ms examinando un solo documento. **Sin
índice**, la diferencia es drástica: escanear `bench_embedded` (documentos
grandes, con la secuencia completa y los 6 perfiles de k-mers) toma ~3 segundos,
mientras que escanear `bench_kmers_alt` (documentos pequeños, solo k-mers) toma
apenas ~0.03 segundos — casi 90 veces más rápido. Esto confirma por qué el índice
sobre `original_id`/`sequence_id` es imprescindible en el modelo embebido: sin
él, cada búsqueda puntual paga el costo de leer documentos mucho más pesados.

### Consulta 2 — Suma global de un k-mer específico en toda la muestra
(agregación `$group`/`$sum`)

In [5]:
print(json.dumps({k: v for k, v in qr.items() if k.startswith("Q2")}, indent=2))

{
  "Q2_suma_global_kmer_embebido_s": 0.33119,
  "Q2_suma_global_kmer_aparte_s": 0.33089
}


**Interpretación:** ambas estrategias deben recorrer toda la colección (no
hay índice que ayude a una suma global), por lo que el tiempo es comparable; la
colección aparte es ligeramente más rápida por tener documentos más pequeños
(menos I/O por documento leído).

### Consulta 3 — Metadatos + perfil de k-mers de una secuencia (caso de uso combinado)

In [6]:
print(json.dumps({k: v for k, v in qr.items() if k.startswith("Q3")}, indent=2))

{
  "Q3_metadatos_y_kmers_embebido_s": 0.00195,
  "Q3_metadatos_y_kmers_aparte_s": 0.00143
}


**Interpretación:** el modelo embebido resuelve esto con **una sola
consulta** (un solo round-trip a MongoDB); el modelo de colección aparte necesita
**dos consultas secuenciales** (metadatos, luego k-mers por `sequence_id`). Para
este caso de uso — que es, de hecho, el más común en el proyecto (consultar una
secuencia y su perfil de k-mers juntos, consulta 15) — el modelo embebido es
operacionalmente más simple, aunque la diferencia de tiempo medida aquí sea
pequeña a esta escala.

## Conclusión de la comparación

Para el patrón de acceso de este proyecto —donde los k-mers casi siempre se
consultan junto con su secuencia dueña (consultas 8, 12, 15) y nunca se
comparten entre secuencias— el modelo **embebido** es la elección correcta: es
operacionalmente más simple (una consulta en vez de dos) y, con el índice
correspondiente, su rendimiento puntual es equivalente al de una colección
aparte. La colección aparte solo mostraría ventaja real en un escenario distinto:
si se necesitara escanear/agregar k-mers con mucha frecuencia **sin** tocar las
secuencias (por ejemplo, un servicio que solo calcula estadísticas de k-mers),
donde evitar cargar el campo `sequence` de cada documento ahorraría I/O.

In [7]:
# Limpieza de las colecciones de benchmark (no son parte del modelo de produccion)
for c in ("bench_embedded", "bench_meta", "bench_kmers_alt"):
    db_drop = exp.db[c].drop()
print("Colecciones de benchmark eliminadas.")

Colecciones de benchmark eliminadas.
